# Load-balanced Gini: fixing the polysemantic hub-feature problem

`feature_purity_analysis.py` found that Gini's per-feature causal importance is heavily concentrated (67% of importance in the top 10% of features on Fashion-MNIST) on features that are essentially maximally class-promiscuous (purity ~0.001) -- i.e. a few polysemantic "hub" directions, not genuine monosemantic specialization. `gini_load_balanced.py` adds an MoE-style batch-level load-balancing penalty alongside the existing per-sample Gini term to discourage the same features dominating every input.

This notebook sweeps the balance-penalty strength (`lambda_balance`) and reports sparsity, MSE, importance concentration, and purity for each value, so we can see whether it fixes the pathology without destroying sparsity or reconstruction quality.

**Before running:** Runtime -> Change runtime type -> GPU.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (go to Runtime > Change runtime type > GPU)')

In [ ]:
!git clone https://github.com/willkn/SAE-Gini.git
%cd SAE-Gini/experiments

In [ ]:
# lambda_balance=0.0 is the control (reproduces the original broken Gini).
# First pass found lambda=0.01 as a strong sweet spot on purity/concentration
# (matched sparsity, similar MSE, purity beat TopK's) but that run didn't check
# whether the original steering-impact advantage over TopK survived the fix --
# concentration itself was what drove the raw ablate/clamp numbers, so fixing
# it may have shrunk or erased that advantage. This run reports ablate/clamp
# steering impact alongside purity/concentration for each lambda, with a finer
# sweep around 0.01.
!python gini_load_balanced.py --dataset fashion_mnist --seed 0 --lambdas 0.0 0.002 0.005 0.01 0.02 0.05

In [ ]:
import json
with open('results/gini_load_balance/fashion_mnist_seed0.json') as f:
    results = json.load(f)

print(f"{'Model':16s} {'MSE':>8s} {'Sparsity':>9s} {'ImpGini':>8s} {'Top10Share':>11s} "
      f"{'MeanPurity':>11s} {'Top10Purity':>12s} {'Ablate':>8s} {'Clamp':>8s}")
for name, r in results.items():
    print(f"{name:16s} {r['mse']:8.4f} {r['relative_sparsity']:9.3f} "
          f"{r['importance_gini_coefficient']:8.3f} {r['top10_importance_share']:11.3f} "
          f"{r['mean_purity']:11.3f} {r['mean_purity_top10pct_by_importance']:12.3f} "
          f"{r['steering_impact_ablate']:8.4f} {r['steering_impact_clamp']:8.4f}")
results

In [ ]:
!zip -r gini_load_balance_results.zip results/gini_load_balance results/feature_purity
from google.colab import files
files.download('gini_load_balance_results.zip')